In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve

# ==========================================
# 1. 디렉토리 설정
# ==========================================
DATA_DIR = "./data/data/processed"
MODEL_DIR = "./model/machine_learning"
TEST_DIR = "./test"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("=" * 50)
print("🚀 [ML] 데이터 로드 및 전처리 시작")
print("=" * 50)

# ==========================================
# 2. 정형 데이터 로드
# ==========================================
csv_path = os.path.join(DATA_DIR, "customer_features_20181101_v1.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(DATA_DIR, "train_20181101_v1.csv")

df_ml = pd.read_csv(csv_path)
print(f"정형 데이터 shape: {df_ml.shape}")

# 타겟 변수 동적 설정
target_col = "churn" if "churn" in df_ml.columns else ("is_churn" if "is_churn" in df_ml.columns else df_ml.columns[-1])
y = df_ml[target_col].values

# 피처 분리
drop_cols = [target_col, "userId", "page_sequence", "session_item_sequence", "padded_sequence"]
feature_cols = [col for col in df_ml.columns if col not in drop_cols]
X_tab = df_ml[feature_cols].copy()

# 범주형 및 수치형 타입 정제
cat_cols = ["gender", "level", "last_page"]
for col in cat_cols:
    if col in X_tab.columns:
        X_tab[col] = X_tab[col].astype("category")

for col in X_tab.columns:
    if X_tab[col].dtype == "object" and col not in cat_cols:
        X_tab[col] = pd.to_numeric(X_tab[col], errors="coerce").fillna(0.0)

# ==========================================
# 3. Data Split
# ==========================================
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)

X_tab_train, X_tab_test = X_tab.iloc[train_idx], X_tab.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# ==========================================
# 4. LightGBM 모델 학습 및 평가
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] LightGBM 모델 학습 시작")
print("=" * 50)

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "max_depth": 3,
    "num_leaves": 7,
    "min_child_samples": 50,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "n_estimators": 500,
    "random_state": 42,
    "verbose": -1,
}

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_tab_train,
    y_train,
    eval_set=[(X_tab_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)],
)

lgb_preds_proba = lgb_model.predict_proba(X_tab_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_preds_proba)
print(f"\n🚀 [LightGBM 단독] Test ROC-AUC Score: {lgb_auc:.4f}")

# LightGBM 임계값 최적화 및 평가
lgb_precisions, lgb_recalls, lgb_thresholds = precision_recall_curve(y_test, lgb_preds_proba)
lgb_f1_scores = 2 * (lgb_precisions * lgb_recalls) / (lgb_precisions + lgb_recalls + 1e-10)
lgb_best_threshold = lgb_thresholds[np.argmax(lgb_f1_scores)]

print(f"🏆 LightGBM F1-Score 기준 최적 임계값: {lgb_best_threshold:.4f}")
print(classification_report(y_test, (lgb_preds_proba >= lgb_best_threshold).astype(int)))

# 모델 저장
joblib.dump(lgb_model, os.path.join(MODEL_DIR, "lgb_model.pkl"))
np.save(os.path.join(TEST_DIR, "lgb_preds_proba.npy"), lgb_preds_proba)
print("✅ LightGBM 모델 및 예측값 저장 완료!")


# ==========================================
# 5. CatBoost 모델 학습 및 평가
# ==========================================
print("\n" + "=" * 50)
print("🚀 [ML] CatBoost 모델 학습 시작")
print("=" * 50)

# CatBoost 범주형 컬럼 인덱스 추출
cat_feature_indices = [X_tab.columns.get_loc(col) for col in cat_cols if col in X_tab.columns]

cb_params = {
    "iterations": 1000,
    "learning_rate": 0.03,
    "depth": 4,
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 100,
}

cb_model = CatBoostClassifier(**cb_params)
cb_model.fit(
    X_tab_train,
    y_train,
    eval_set=(X_tab_test, y_test),
    cat_features=cat_feature_indices,
    early_stopping_rounds=50,
    use_best_model=True
)

cb_preds_proba = cb_model.predict_proba(X_tab_test)[:, 1]
cb_auc = roc_auc_score(y_test, cb_preds_proba)
print(f"\n🚀 [CatBoost 단독] Test ROC-AUC Score: {cb_auc:.4f}")

# CatBoost 임계값 최적화 및 평가
cb_precisions, cb_recalls, cb_thresholds = precision_recall_curve(y_test, cb_preds_proba)
cb_f1_scores = 2 * (cb_precisions * cb_recalls) / (cb_precisions + cb_recalls + 1e-10)
cb_best_threshold = cb_thresholds[np.argmax(cb_f1_scores)]

print(f"🏆 CatBoost F1-Score 기준 최적 임계값: {cb_best_threshold:.4f}")
print(classification_report(y_test, (cb_preds_proba >= cb_best_threshold).astype(int)))

# 모델 저장 (CatBoost 전용 save_model 및 예측값 저장)
cb_model.save_model(os.path.join(MODEL_DIR, "catboost_model.cbm"))
np.save(os.path.join(TEST_DIR, "cb_preds_proba.npy"), cb_preds_proba)
print("✅ CatBoost 모델 및 예측값 저장 완료!")

🚀 [ML] 데이터 로드 및 전처리 시작
정형 데이터 shape: (18139, 16)

🚀 [ML] LightGBM 모델 학습 시작
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.715069


c:\Users\playdata2\miniconda3\envs\agent\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's auc: 0.715651
Early stopping, best iteration is:
[153]	valid_0's auc: 0.716271

🚀 [LightGBM 단독] Test ROC-AUC Score: 0.7163
🏆 LightGBM F1-Score 기준 최적 임계값: 0.1175
              precision    recall  f1-score   support

           0       0.95      0.94      0.94      3421
           1       0.20      0.27      0.23       207

    accuracy                           0.90      3628
   macro avg       0.58      0.60      0.59      3628
weighted avg       0.91      0.90      0.90      3628

✅ LightGBM 모델 및 예측값 저장 완료!

🚀 [ML] CatBoost 모델 학습 시작
0:	test: 0.6207609	best: 0.6207609 (0)	total: 135ms	remaining: 2m 14s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7090759404
bestIteration = 3

Shrink model to first 4 iterations.

🚀 [CatBoost 단독] Test ROC-AUC Score: 0.7091
🏆 CatBoost F1-Score 기준 최적 임계값: 0.4378
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      3421
           1       0.19      0.27      0.22     